In [ ]:
# ============================================================
# CELL 1 — Imports & settings
# Run once. Edit settings here before starting.
# ============================================================

import numpy as np
import open3d as o3d
import os, sys
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d

# ── File & frame ──────────────────────────────────────────
INPUT_FILE  = r'C:\ROSBAGS VERWIJDER NA BEP\14_20_00\rosbag\rosbag_0.mcap'


# ── Sensor topics ─────────────────────────────────────────
LIDAR_TOPICS = [
    "/rslidar/M1P_deskewed",
    "/rslidar/helios_R",
    "/rslidar/helios_L",
]

# ── Sensor transforms (from Foxglove TF tree) ─────────────
SENSOR_TRANSFORMS = {
    "/rslidar/M1P_deskewed":  {"translation": [ 0.800,  0.000, 0.876], "rotation_rpy_deg": [0.1, -1.7,  1.7]},
    "/rslidar/helios_R":      {"translation": [-0.743, -0.243, 0.857], "rotation_rpy_deg": [2.4, -2.6, -179.9]},
    "/rslidar/helios_L":      {"translation": [-0.745,  0.191, 0.876], "rotation_rpy_deg": [3.5, -2.1,  179.9]},
}

# ── CSF parameters ────────────────────────────────────────
CLOTH_RESOLUTION = 0.5
SLOPE_SMOOTH     = True
THRESHOLD        = 0.3

# ── Profile parameters ────────────────────────────────────
Y_SLICE       = 0.5    # ±meters from centerline to include
BIN_SIZE      = 0.2    # meters per bin along X
SMOOTH_WINDOW = 8      # bins to smooth over

# ── Output ────────────────────────────────────────────────
OUTPUT_PLOT = r'C:\ROSBAGS VERWIJDER NA BEP\ground_profile.png'

print("Settings loaded.")


In [ ]:
# ============================================================
# CELL 1b — Frame estimator from timestamp
# Reads actual timestamps from the mcap file automatically.
# Run once after Cell 1, then rerun whenever you change timestamp.
# ============================================================

from mcap.reader import make_reader
from mcap_ros2.decoder import DecoderFactory

TARGET_TIMESTAMP = 1777638090.739138180  # <-- paste your Foxglove timestamp here

REFERENCE_TOPIC = "/rslidar/M1P_deskewed"  # topic to use for frame counting

print(f"Scanning timestamps from '{REFERENCE_TOPIC}'...")
print("(This may take a moment...)")

timestamps = []
with open(INPUT_FILE, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(topics=[REFERENCE_TOPIC]):
        # log_time is in nanoseconds — convert to seconds
        timestamps.append(message.log_time / 1e9)

timestamps = np.array(timestamps)
print(f"Found {len(timestamps)} frames")
print(f"Recording start: {timestamps[0]:.3f}")
print(f"Recording end:   {timestamps[-1]:.3f}")
print(f"Duration:        {timestamps[-1] - timestamps[0]:.2f}s")

# Find closest frame to target timestamp
diffs          = np.abs(timestamps - TARGET_TIMESTAMP)
frame_estimate = int(np.argmin(diffs))
closest_time   = timestamps[frame_estimate]
time_error     = abs(closest_time - TARGET_TIMESTAMP)

print(f"\nTarget timestamp:  {TARGET_TIMESTAMP:.3f}")
print(f"Closest timestamp: {closest_time:.3f}  (error: {time_error*1000:.1f}ms)")
print(f"→ Estimated frame: {frame_estimate}")

# Auto-update FRAME_INDEX
FRAME_INDEX = frame_estimate
print(f"\nFRAME_INDEX updated to {FRAME_INDEX} — rerun from Cell 3")

In [ ]:
# ============================================================
# CELL 2 — Helper functions
# Run once.
# ============================================================

def rpy_deg_to_rotation_matrix(roll_deg, pitch_deg, yaw_deg):
    r, p, y = np.radians(roll_deg), np.radians(pitch_deg), np.radians(yaw_deg)
    Rx = np.array([[1,0,0],[0,np.cos(r),-np.sin(r)],[0,np.sin(r),np.cos(r)]])
    Ry = np.array([[np.cos(p),0,np.sin(p)],[0,1,0],[-np.sin(p),0,np.cos(p)]])
    Rz = np.array([[np.cos(y),-np.sin(y),0],[np.sin(y),np.cos(y),0],[0,0,1]])
    return Rz @ Ry @ Rx

def transform_points(points, translation, rotation_rpy_deg):
    R = rpy_deg_to_rotation_matrix(*rotation_rpy_deg)
    return (R @ points.T).T + np.array(translation)

def _ros_pc2_to_numpy(msg):
    import struct
    field_map = {f.name: f for f in msg.fields}
    if not all(k in field_map for k in ("x","y","z")):
        return None
    x_off, y_off, z_off = field_map["x"].offset, field_map["y"].offset, field_map["z"].offset
    step, data, n = msg.point_step, msg.data, msg.width * msg.height
    xyz = np.zeros((n, 3), dtype=np.float32)
    for i in range(n):
        b = i * step
        xyz[i,0] = struct.unpack_from("f", data, b + x_off)[0]
        xyz[i,1] = struct.unpack_from("f", data, b + y_off)[0]
        xyz[i,2] = struct.unpack_from("f", data, b + z_off)[0]
    return xyz[np.isfinite(xyz).all(axis=1)]

def load_mcap_topic(filepath, topic, frame_index):
    from mcap.reader import make_reader
    from mcap_ros2.decoder import DecoderFactory
    frames = []
    with open(filepath, "rb") as f:
        reader = make_reader(f, decoder_factories=[DecoderFactory()])
        for schema, channel, message, ros_msg in reader.iter_decoded_messages(topics=[topic]):
            pts = _ros_pc2_to_numpy(ros_msg)
            if pts is not None and len(pts) > 0:
                frames.append(pts)
    if not frames:
        print(f"  WARNING: No messages on '{topic}' — skipping.")
        return None
    idx = min(int(frame_index), len(frames) - 1)
    print(f"  [{topic}]  frame {idx}/{len(frames)-1}  ({len(frames[idx]):,} points)")
    return frames[idx]

print("Helper functions defined.")


In [ ]:
# ============================================================
# CELL 3 — Load raw point clouds from mcap
# SLOW — only rerun if you change FRAME_INDEX or INPUT_FILE
# ============================================================

print(f"Loading frame {FRAME_INDEX} from {len(LIDAR_TOPICS)} sensors...")
raw_clouds = {}
for topic in LIDAR_TOPICS:
    pts = load_mcap_topic(INPUT_FILE, topic, FRAME_INDEX)
    if pts is not None:
        raw_clouds[topic] = pts

print(f"\nLoaded {len(raw_clouds)} sensors.")


In [ ]:
# ============================================================
# CELL 4 — Apply sensor transforms & merge into one cloud
# Rerun if you change SENSOR_TRANSFORMS
# ============================================================

transformed = {}
for topic, pts in raw_clouds.items():
    tf  = SENSOR_TRANSFORMS[topic]
    transformed[topic] = transform_points(pts, tf["translation"], tf["rotation_rpy_deg"])

merged = np.vstack(list(transformed.values()))
print(f"Merged cloud: {len(merged):,} points")
print(f"  X  {merged[:,0].min():.2f} → {merged[:,0].max():.2f} m")
print(f"  Y  {merged[:,1].min():.2f} → {merged[:,1].max():.2f} m")
print(f"  Z  {merged[:,2].min():.2f} → {merged[:,2].max():.2f} m")


In [ ]:
# ============================================================
# CELL 5 — CSF ground filter
# Rerun if you change CLOTH_RESOLUTION, SLOPE_SMOOTH, THRESHOLD
# ============================================================

import CSF

csf = CSF.CSF()
csf.params.bSloopSmooth     = SLOPE_SMOOTH
csf.params.cloth_resolution = CLOTH_RESOLUTION
csf.params.threshold        = THRESHOLD
csf.setPointCloud(merged)

ground_idx     = CSF.VecInt()
non_ground_idx = CSF.VecInt()
csf.do_filtering(ground_idx, non_ground_idx)

ground     = merged[np.array(list(ground_idx))]
non_ground = merged[np.array(list(non_ground_idx))]

pct_g  = 100 * len(ground) / len(merged)
pct_ng = 100 * len(non_ground) / len(merged)
print(f"Ground points:     {len(ground):,}  ({pct_g:.1f}%)")
print(f"Non-ground points: {len(non_ground):,}  ({pct_ng:.1f}%)")


In [ ]:
# ============================================================
# CELL 6 — Slice along centerline & bin by X
# Rerun if you change Y_SLICE or BIN_SIZE
# ============================================================

# Slice to centerline strip
mask      = np.abs(ground[:, 1]) <= Y_SLICE
slice_pts = ground[mask]
print(f"Points within Y ±{Y_SLICE}m: {len(slice_pts):,}")

if len(slice_pts) < 10:
    print("ERROR: Too few points. Try increasing Y_SLICE.")
else:
    x = -slice_pts[:, 0]
    z = slice_pts[:, 2]

    # Bin by X
    x_min = np.floor(x.min() / BIN_SIZE) * BIN_SIZE
    x_max = np.ceil(x.max()  / BIN_SIZE) * BIN_SIZE
    bins  = np.arange(x_min, x_max + BIN_SIZE, BIN_SIZE)

    bin_centers, bin_heights = [], []
    for i in range(len(bins) - 1):
        in_bin = (x >= bins[i]) & (x < bins[i+1])
        if in_bin.sum() >= 3:
            bin_centers.append((bins[i] + bins[i+1]) / 2)
            bin_heights.append(np.median(z[in_bin]))

    bin_centers = np.array(bin_centers)
    bin_heights = np.array(bin_heights)
    print(f"Bins with data:   {len(bin_centers)}")
    print(f"X range:          {bin_centers.min():.1f} → {bin_centers.max():.1f} m")


In [ ]:
# ============================================================
# CELL 7 — Compute slope & curvature
# Rerun if you change SMOOTH_WINDOW
# ============================================================

z_smooth         = uniform_filter1d(bin_heights, size=SMOOTH_WINDOW)
dz               = np.gradient(z_smooth, bin_centers)
slope_deg        = np.degrees(np.arctan(dz))
d2z              = np.gradient(dz, bin_centers)
curvature        = d2z
slope_smooth     = uniform_filter1d(slope_deg, size=SMOOTH_WINDOW)
curvature_smooth = uniform_filter1d(curvature, size=SMOOTH_WINDOW)

print(f"Slope range:     {slope_smooth.min():.2f}° → {slope_smooth.max():.2f}°")
print(f"Curvature range: {curvature_smooth.min():.4f} → {curvature_smooth.max():.4f} m⁻¹")


In [ ]:
# ============================================================
# CELL 8 — Plot ground height, slope & curvature
# Rerun freely to tweak the graph appearance
# ============================================================

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle(f"Ground Profile — Frame {FRAME_INDEX}  |  Y slice ±{Y_SLICE}m",
             fontsize=14, fontweight='bold')

# ── Height ────────────────────────────────────────────────
ax1 = axes[0]
ax1.plot(bin_centers, z_smooth, color='royalblue', linewidth=1.5, label='Height (smoothed)')
ax1.scatter(bin_centers, bin_heights, color='lightsteelblue', s=8, alpha=0.5, label='Raw median Z')
ax1.set_ylabel("Height Z (m)", fontsize=11)
ax1.set_title("Ground Height", fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.axhline(0, color='gray', linewidth=0.5, linestyle='--')

# ── Slope ─────────────────────────────────────────────────
ax2 = axes[1]
ax2.plot(bin_centers, slope_smooth, color='darkorange', linewidth=1.5, label='Slope (smoothed)')
ax2.plot(bin_centers, slope_deg,    color='moccasin',   linewidth=0.8, alpha=0.6, label='Raw slope')
ax2.fill_between(bin_centers, slope_smooth, 0, where=(slope_smooth >= 0), alpha=0.15, color='red',  label='Uphill')
ax2.fill_between(bin_centers, slope_smooth, 0, where=(slope_smooth <  0), alpha=0.15, color='blue', label='Downhill')
ax2.set_ylabel("Slope (degrees)", fontsize=11)
ax2.set_title("Ground Slope along Forward Direction", fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.axhline(0, color='gray', linewidth=0.5, linestyle='--')

# ── Curvature ─────────────────────────────────────────────
ax3 = axes[2]
ax3.plot(bin_centers, curvature_smooth, color='green',      linewidth=1.5, label='Curvature (smoothed)')
ax3.plot(bin_centers, curvature,        color='lightgreen', linewidth=0.8, alpha=0.6, label='Raw curvature')
ax3.fill_between(bin_centers, curvature_smooth, 0, where=(curvature_smooth >= 0), alpha=0.15, color='orange', label='Convex (crest)')
ax3.fill_between(bin_centers, curvature_smooth, 0, where=(curvature_smooth <  0), alpha=0.15, color='purple', label='Concave (dip)')
ax3.set_ylabel("Curvature (m⁻¹)", fontsize=11)
ax3.set_xlabel("Forward distance X (m)", fontsize=11)
ax3.set_title("Vertical Curvature along Forward Direction", fontsize=11)
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)
ax3.axhline(0, color='gray', linewidth=0.5, linestyle='--')

plt.tight_layout()
plt.savefig(OUTPUT_PLOT, dpi=150, bbox_inches='tight')
print(f"Saved to: {OUTPUT_PLOT}")
plt.show()
